In [3]:
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

RANDOM_STATE = 42
DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SUBMISSION_PATH = DATA_DIR / 'submission.csv'


def parse_timestamp_to_minutes(timestamp_series: pd.Series) -> pd.Series:
    '''Convert HH:MM strings into minutes past midnight.'''
    parts = timestamp_series.astype(str).str.split(':', n=1, expand=True)
    hours = parts[0].astype(int)
    minutes = parts[1].astype(int)
    return hours * 60 + minutes



def decode_single_geohash(geohash_value: str) -> tuple[float, float]:
    '''Decode one geohash string into latitude and longitude.'''
    if pd.isna(geohash_value):
        return np.nan, np.nan

    try:
        decoded = pgh.decode_exactly(str(geohash_value))
        if hasattr(decoded, 'latitude') and hasattr(decoded, 'longitude'):
            return float(decoded.latitude), float(decoded.longitude)
        return float(decoded[0]), float(decoded[1])
    except Exception:
        try:
            decoded = pgh.decode(str(geohash_value))
            if hasattr(decoded, 'latitude') and hasattr(decoded, 'longitude'):
                return float(decoded.latitude), float(decoded.longitude)
            return float(decoded[0]), float(decoded[1])
        except Exception:
            return np.nan, np.nan



def add_geohash_coordinates(frame: pd.DataFrame) -> pd.DataFrame:
    '''Add latitude and longitude columns derived from geohash values.'''
    geohash_series = frame['geohash'].astype('string')
    unique_geohashes = geohash_series.dropna().unique()
    geohash_lookup = {value: decode_single_geohash(value) for value in unique_geohashes}

    coordinates = geohash_series.map(geohash_lookup)
    frame['latitude'] = [pair[0] if isinstance(pair, tuple) else np.nan for pair in coordinates]
    frame['longitude'] = [pair[1] if isinstance(pair, tuple) else np.nan for pair in coordinates]
    return frame

In [4]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

train_raw['is_test'] = False
test_raw['is_test'] = True
test_raw['demand'] = np.nan

combined_df = pd.concat([train_raw, test_raw], ignore_index=True, sort=False)
combined_df['time_mins'] = parse_timestamp_to_minutes(combined_df['timestamp']).astype(int)
combined_df['time_sin'] = np.sin(2 * np.pi * combined_df['time_mins'] / 1440.0)
combined_df['time_cos'] = np.cos(2 * np.pi * combined_df['time_mins'] / 1440.0)
combined_df = add_geohash_coordinates(combined_df)

combined_df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,is_test,time_mins,time_sin,time_cos,latitude,longitude
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN,False,0,0.0,1.0,-5.484924,90.664673
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,False,0,0.0,1.0,-5.462952,90.686646
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,False,0,0.0,1.0,-5.462952,90.708618
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,False,0,0.0,1.0,-5.462952,90.862427
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,False,0,0.0,1.0,-5.457458,90.675659


In [5]:
combined_df = combined_df.sort_values(['geohash', 'day', 'time_mins'], kind='mergesort').reset_index(drop=True)

combined_df['Temperature'] = combined_df.groupby('geohash')['Temperature'].transform(lambda series: series.ffill().bfill())
combined_df['Weather'] = combined_df.groupby('geohash')['Weather'].transform(lambda series: series.ffill().bfill())
combined_df['RoadType'] = combined_df['RoadType'].fillna('Unknown')

combined_df['demand_24h_ago'] = combined_df.groupby('geohash')['demand'].shift(96)

categorical_columns = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for column_name in categorical_columns:
    combined_df[column_name] = combined_df[column_name].astype('category')

combined_df.dtypes

Index                int64
geohash           category
day                  int64
timestamp           object
demand             float64
RoadType          category
NumberofLanes        int64
LargeVehicles     category
Landmarks         category
Temperature        float64
Weather           category
is_test               bool
time_mins            int32
time_sin           float64
time_cos           float64
latitude           float64
longitude          float64
demand_24h_ago     float64
dtype: object

In [ ]:
train_df = combined_df.loc[~combined_df['is_test']].copy()
test_df = combined_df.loc[combined_df['is_test']].copy()

feature_columns = [
    'geohash',
    'day',
    'time_mins',
    'time_sin',
    'time_cos',
    'latitude',
    'longitude',
    'RoadType',
    'NumberofLanes',
    'LargeVehicles',
    'Landmarks',
    'Temperature',
    'Weather',
    'demand_24h_ago',
]

train_mask = (train_df['day'] == 48) & (train_df['time_mins'] <= 720)
valid_mask = (train_df['day'] == 48) & (train_df['time_mins'] > 720)

X_train = train_df.loc[train_mask, feature_columns].copy()
y_train = train_df.loc[train_mask, 'demand'].astype(float)
X_valid = train_df.loc[valid_mask, feature_columns].copy()
y_valid = train_df.loc[valid_mask, 'demand'].astype(float)


def build_regressor(device_type: str, n_estimators: int) -> lgb.LGBMRegressor:
    '''Create a LightGBM regressor with the requested accelerator mode.'''
    return lgb.LGBMRegressor(
        objective='regression',
        random_state=RANDOM_STATE,
        n_estimators=n_estimators,
        learning_rate=0.05,
        device_type=device_type,
        n_jobs=-1,
        verbosity=-1,
    )


try:
    model = build_regressor('cuda', 1000)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='rmse',
        categorical_feature=categorical_columns,
        callbacks=[lgb.early_stopping(50, verbose=True)],
    )
    trained_device = 'cuda'
except lgb.basic.LightGBMError as error:
    if 'CUDA Tree Learner was not enabled' not in str(error):
        raise
    print('CUDA build is unavailable in this environment. Falling back to CPU training.')
    model = build_regressor('cpu', 1000)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='rmse',
        categorical_feature=categorical_columns,
        callbacks=[lgb.early_stopping(50, verbose=True)],
    )
    trained_device = 'cpu'

valid_predictions = model.predict(X_valid, num_iteration=model.best_iteration_)
valid_r2 = r2_score(y_valid, valid_predictions)
print(f'Validation R2: {valid_r2:.6f}')
print(f'Validation Score: {max(0.0, 100 * valid_r2):.4f}')
print(f'Training device used: {trained_device}')

best_n_estimators = model.best_iteration_ or 1000
full_train_df = train_df.loc[train_df['demand'].notna()].copy()
X_full_train = full_train_df[feature_columns].copy()
y_full_train = full_train_df['demand'].astype(float)

final_model = build_regressor(trained_device, best_n_estimators)
final_model.fit(
    X_full_train,
    y_full_train,
    categorical_feature=categorical_columns,
)

test_predictions = final_model.predict(test_df[feature_columns])
submission = pd.DataFrame({
    'Index': test_df['Index'].astype(int),
    'demand': test_predictions,
})
submission.to_csv(SUBMISSION_PATH, index=False)
submission.head()

LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1